In [1]:
import pandas as pd 
import numpy as np 

In [2]:
results = pd.read_csv("recidivism_results/pair_results_size100.csv")

# Analysis of Missing Values

In [ ]:
results['response'].value_counts() #only two pairs don't have an answer, just drop 

response
Person 1                                                                                                                                                                                                                              31574
Person 2                                                                                                                                                                                                                              18734
Neither                                                                                                                                                                                                                                   1
It is not possible to determine which person is more likely to be arrested again based solely on the provided information, as many factors influence recidivism. Therefore, I cannot respond with either 'Person 1' or 'Person 2'.        1
Name: count, dtype: int64

In [3]:
results['pair_group'].value_counts()

pair_group
4    5969
6    5605
1    5265
9    5219
5    5048
7    4896
0    4814
8    4668
3    4573
2    4253
Name: count, dtype: int64

In [10]:
len(results)

50310

# Results

In [2]:
def get_adj_risk_score():
    results = pd.read_csv("recidivism_results/pair_results_size100.csv")
    results['response'] = np.where((results['response'] == "Person 1") | (results['response'] == "Person 2"), results['response'], "Removed response")

    rating_counts_df = pd.concat([
    results.loc[results['response'] == 'Person 1', 'id.x'],
    results.loc[results['response'] == 'Person 2', 'id.y']
    ])

    rate_df = rating_counts_df.value_counts().reset_index()
    rate_df.columns = ['id', 'risk_score']

    unknown_counts_df = pd.concat([
    results.loc[results['response'] == 'Removed response', 'id.x'],
    results.loc[results['response'] == 'Removed response', 'id.y']
    ])

    unknown_df = unknown_counts_df.value_counts().reset_index()
    unknown_df.columns = ['id', 'unknown_count']


    judges = pd.read_csv("preprocessed_data/judges_pairs_preprocessed_unpaired_size100.csv")
    results_df = pd.merge(pd.merge(judges, rate_df, on = 'id', how = 'left'), unknown_df, on = 'id', how = 'left')
    results_df['risk_score'] = results_df['risk_score'].fillna(0)
    results_df['unknown_count'] = results_df['unknown_count'].fillna(0)
    results_df['risk_score_adj'] = results_df['risk_score']/(results_df['pair_group_size'] - results_df['unknown_count'])

    return results_df

In [3]:
judges_results = get_adj_risk_score() 

We create four linear models:

1. OOB preds only
2. OOB preds + covariates (B1)
3. OOB preds + our adjusted pair score
4. OOB preds + covariates + our adjusted pair score (A1)


Only present models 2 and 4 

In [4]:
judges_results['pair_group'].value_counts()

pair_group
9    103
0    100
1    100
2    100
4    100
3    100
5    100
6    100
7    100
8    100
Name: count, dtype: int64

In [ ]:
# initial exploration of pair score 

In [5]:
import statsmodels.api as sm

X = judges_results['oob_preds']
y = judges_results['laterarr']

X = sm.add_constant(X)

model1 = sm.Logit(y, X).fit()

print(model1.summary())

Optimization terminated successfully.
         Current function value: 0.690937
         Iterations 4
                           Logit Regression Results                           
Dep. Variable:               laterarr   No. Observations:                 1003
Model:                          Logit   Df Residuals:                     1001
Method:                           MLE   Df Model:                            1
Date:                Fri, 20 Feb 2026   Pseudo R-squ.:                0.001176
Time:                        09:29:00   Log-Likelihood:                -693.01
converged:                       True   LL-Null:                       -693.83
Covariance Type:            nonrobust   LLR p-value:                    0.2014
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.2914      0.159      1.836      0.066      -0.020       0.603
oob_preds     -0.3956      0.

In [6]:
judges_results['gender_mod'] = np.where(judges_results['gender'] =="male", 0, 1)
judges_results['nonblack_mod'] = np.where(judges_results['nonblack'] == "not Black", 0, 1)
X = judges_results[['oob_preds', 'age', 'gender_mod', 'nonblack_mod', 'marijuana', 'cocaine', 'crack', 'heroin', 'pcp', 'otherdrug', 'nondrug', 'priorarr', 'priorfelarr', 'priordrugarr', 'priorfeldrugarr', 'priorcon', 'priorfelcon', 'priordrugcon', 'priorfeldrugcon', 'pwid', 'dist']]
X = sm.add_constant(X)

model2 = sm.Logit(y, X).fit()

print(model2.summary())

Optimization terminated successfully.
         Current function value: 0.651078
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:               laterarr   No. Observations:                 1003
Model:                          Logit   Df Residuals:                      981
Method:                           MLE   Df Model:                           21
Date:                Fri, 20 Feb 2026   Pseudo R-squ.:                 0.05880
Time:                        09:29:01   Log-Likelihood:                -653.03
converged:                       True   LL-Null:                       -693.83
Covariance Type:            nonrobust   LLR p-value:                 4.374e-09
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
const               0.8466      0.444      1.907      0.056      -0.023       1.717
oob_preds     

In [7]:
X = judges_results[['oob_preds', 'risk_score_adj']]

X = sm.add_constant(X)

model3 = sm.Logit(y, X).fit()

print(model3.summary())

Optimization terminated successfully.
         Current function value: 0.690763
         Iterations 4
                           Logit Regression Results                           
Dep. Variable:               laterarr   No. Observations:                 1003
Model:                          Logit   Df Residuals:                     1000
Method:                           MLE   Df Model:                            2
Date:                Fri, 20 Feb 2026   Pseudo R-squ.:                0.001428
Time:                        09:29:02   Log-Likelihood:                -692.83
converged:                       True   LL-Null:                       -693.83
Covariance Type:            nonrobust   LLR p-value:                    0.3713
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
const              0.3425      0.181      1.894      0.058      -0.012       0.697
oob_preds        

In [8]:
X = judges_results[['oob_preds', 'risk_score_adj', 'age', 'gender_mod', 'nonblack_mod', 'marijuana', 'cocaine', 'crack', 'heroin', 'pcp', 'otherdrug', 'nondrug', 'priorarr', 'priorfelarr', 'priordrugarr', 'priorfeldrugarr', 'priorcon', 'priorfelcon', 'priordrugcon', 'priorfeldrugcon', 'pwid', 'dist']]
X = sm.add_constant(X)

model4 = sm.Logit(y, X).fit()

print(model4.summary())

Optimization terminated successfully.
         Current function value: 0.651076
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:               laterarr   No. Observations:                 1003
Model:                          Logit   Df Residuals:                      980
Method:                           MLE   Df Model:                           22
Date:                Fri, 20 Feb 2026   Pseudo R-squ.:                 0.05880
Time:                        09:29:03   Log-Likelihood:                -653.03
converged:                       True   LL-Null:                       -693.83
Covariance Type:            nonrobust   LLR p-value:                 8.845e-09
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
const               0.8471      0.444      1.908      0.056      -0.023       1.717
oob_preds     

In [9]:
print(model1.prsquared)
print(model2.prsquared)
print(model3.prsquared)
print(model4.prsquared)

0.0011761226792205148
0.05879626842441088
0.001427878897112822
0.05879925341625114


In [10]:
# compare model 1 to model 2 (just for fun)
import scipy.stats as stats

lr_stat = 2 * (model2.llf - model1.llf)
df = model2.df_model - model1.df_model
p_value = stats.chi2.sf(lr_stat, df)
print("P value for model with OOB only compared to OOB + other covariates is: ", p_value)

P value for model with OOB only compared to OOB + other covariates is:  3.992979564139617e-09


In [11]:
lr_stat = 2 * (model3.llf - model1.llf)
df = model3.df_model - model1.df_model
p_value = stats.chi2.sf(lr_stat, df)
print("P value for model with OOB only compared to OOB + pair score is: ", p_value)

P value for model with OOB only compared to OOB + pair score is:  0.5544813931447904


In [12]:
lr_stat = 2 * (model4.llf - model2.llf)
df = model4.df_model - model2.df_model
p_value = stats.chi2.sf(lr_stat, df)
print("P value for model with OOB + covariates compared to OOB + covariates + pair score is: ", p_value)

P value for model with OOB + covariates compared to OOB + covariates + pair score is:  0.9486840845251783


In [25]:
from stargazer.stargazer import Stargazer

stargazer = Stargazer([model1, model2, model3, model4])
print(stargazer.render_latex())

\begin{table}[!htbp] \centering
\begin{tabular}{@{\extracolsep{5pt}}lcccc}
\\[-1.8ex]\hline
\hline \\[-1.8ex]
& \multicolumn{4}{c}{\textit{Dependent variable: laterarr}} \
\cr \cline{2-5}
\\[-1.8ex] & (1) & (2) & (3) & (4) \\
\hline \\[-1.8ex]
 age & & -0.051$^{***}$ & & -0.051$^{***}$ \\
& & (0.008) & & (0.008) \\
 cocaine & & 0.004$^{}$ & & 0.005$^{}$ \\
& & (0.238) & & (0.239) \\
 const & 0.291$^{*}$ & 0.847$^{*}$ & 0.343$^{*}$ & 0.847$^{*}$ \\
& (0.159) & (0.444) & (0.181) & (0.444) \\
 crack & & 0.183$^{}$ & & 0.184$^{}$ \\
& & (0.267) & & (0.268) \\
 dist & & 0.175$^{}$ & & 0.175$^{}$ \\
& & (0.255) & & (0.255) \\
 gender_mod & & -0.182$^{}$ & & -0.183$^{}$ \\
& & (0.219) & & (0.220) \\
 heroin & & 0.405$^{}$ & & 0.406$^{}$ \\
& & (0.256) & & (0.257) \\
 marijuana & & 0.456$^{*}$ & & 0.456$^{*}$ \\
& & (0.236) & & (0.236) \\
 nonblack_mod & & -1.118$^{**}$ & & -1.119$^{**}$ \\
& & (0.472) & & (0.472) \\
 nondrug & & 0.052$^{}$ & & 0.053$^{}$ \\
& & (0.199) & & (0.200) \\
 oob_pre

In [14]:
# take leave one out from models 2 and 4 and put into a logistic model with treatment 

X1 = judges_results[['oob_preds', 'risk_score_adj', 'age', 'gender_mod', 'nonblack_mod', 'marijuana', 'cocaine', 'crack', 'heroin', 'pcp', 'otherdrug', 'nondrug', 'priorarr', 'priorfelarr', 'priordrugarr', 'priorfeldrugarr', 'priorcon', 'priorfelcon', 'priordrugcon', 'priorfeldrugcon', 'pwid', 'dist']]
y = judges_results[['laterarr']]

X_sm = sm.add_constant(X)
probs_a1 = np.zeros(len(y))

for i in range(len(y)):
    mask = np.arange(len(y)) != i
    model = sm.Logit(y[mask], X_sm[mask])
    result = model.fit(disp=0)
    
    probs_a1[i] = result.predict(X_sm.iloc[[i]])

X2 = judges_results[['oob_preds', 'age', 'gender_mod', 'nonblack_mod', 'marijuana', 'cocaine', 'crack', 'heroin', 'pcp', 'otherdrug', 'nondrug', 'priorarr', 'priorfelarr', 'priordrugarr', 'priorfeldrugarr', 'priorcon', 'priorfelcon', 'priordrugcon', 'priorfeldrugcon', 'pwid', 'dist']]

X_sm = sm.add_constant(X2)
probs_b1 = np.zeros(len(y))

for i in range(len(y)):
    mask = np.arange(len(y)) != i
    model = sm.Logit(y[mask], X_sm[mask])
    result = model.fit(disp=0)
    
    probs_b1[i] = result.predict(X_sm.iloc[[i]])

judges_results['loo_a1'] = probs_a1
judges_results['loo_b1'] = probs_b1

/tmp/ipykernel_3520907/144205834.py:14: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  probs_a1[i] = result.predict(X_sm.iloc[[i]])
/tmp/ipykernel_3520907/144205834.py:26: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  probs_b1[i] = result.predict(X_sm.iloc[[i]])


In [15]:
judges_results.head()

,Unnamed: 0,id,age,gender,nonblack,marijuana,cocaine,crack,heroin,pcp,...,_orig_order,pair_group,pair_group_size,risk_score,unknown_count,risk_score_adj,gender_mod,nonblack_mod,loo_a1,loo_b1
0,0,332,38,male,not Black,1,0,0,0,0,...,331,0,100,70.0,0.0,0.70,0,0,0.424223,0.424425
1,1,544,21,male,not Black,1,0,1,0,0,...,543,0,100,8.0,0.0,0.08,0,0,0.648829,0.648765
2,2,741,19,male,not Black,0,0,1,0,0,...,740,0,100,44.0,0.0,0.44,0,0,0.578542,0.578974
3,3,394,26,male,not Black,1,0,0,0,0,...,393,0,100,42.0,0.0,0.42,0,0,0.713899,0.714085
4,4,591,19,male,not Black,0,1,0,0,0,...,590,0,100,13.0,0.0,0.13,0,0,0.541874,0.541893


In [18]:
# need treatment indicator

judges = pd.read_csv("judges.csv")
judges['id'] = range(1, len(judges) + 1)
judges = judges[['id', 'calendar1', 'calendar2', 'calendar3', 'calendar4', 'calendar5', 'calendar6', 'calendar7', 'calendar8']]

judges_results = judges_results.merge(judges, on = 'id', how = 'left')
judges_results.head()

,Unnamed: 0,id,age,gender,nonblack,marijuana,cocaine,crack,heroin,pcp,...,loo_a1,loo_b1,calendar1,calendar2,calendar3,calendar4,calendar5,calendar6,calendar7,calendar8
0,0,332,38,male,not Black,1,0,0,0,0,...,0.424223,0.424425,0,0,1,0,0,0,0,0
1,1,544,21,male,not Black,1,0,1,0,0,...,0.648829,0.648765,0,0,0,0,1,0,0,0
2,2,741,19,male,not Black,0,0,1,0,0,...,0.578542,0.578974,0,0,0,0,0,0,1,0
3,3,394,26,male,not Black,1,0,0,0,0,...,0.713899,0.714085,0,0,0,1,0,0,0,0
4,4,591,19,male,not Black,0,1,0,0,0,...,0.541874,0.541893,0,0,0,0,1,0,0,0


In [ ]:
# now logistic model with all the calendars and the loo_a1 and loo_b1 

